# AnimalCLEF 2026 - MegaDescriptor LoRA Fine-tuning

這份 notebook 是完全自包含版本。

- 從 `animal-clef-2026.zip` 讀取資料
- 載入 `hf-hub:BVRA/MegaDescriptor-L-384`
- 凍結 backbone 並只訓練 LoRA + projection + classifier
- validation embedding / clustering / ARI
- test submission 產生

注意：第一次跑時需要能連到 Hugging Face 下載模型。

In [1]:
from pathlib import Path
import importlib

cwd = Path.cwd()
print('cwd =', cwd)
print('zip exists =', (cwd / 'animal-clef-2026.zip').exists())

for name in ['torch', 'torchvision', 'timm', 'pandas', 'numpy', 'sklearn', 'PIL', 'tqdm', 'huggingface_hub']:
    mod = importlib.import_module(name)
    print(f'{name}:', getattr(mod, '__version__', 'no_version'))

cwd = c:\Users\User\learn_ML
zip exists = True
torch: 2.5.1+cu121
torchvision: 0.20.1+cu121
timm: 1.0.15
pandas: 2.3.0
numpy: 1.26.4
sklearn: 1.8.0
PIL: 11.3.0
tqdm: 4.67.1
huggingface_hub: 0.34.4


In [2]:
CONFIG = {
    'zip_path': 'animal-clef-2026.zip',
    'output_dir': 'outputs/animal_clef_megadescriptor_lora',
    'backbone': 'hf-hub:BVRA/MegaDescriptor-L-384',
    'image_size': 384,
    'mean': (0.5, 0.5, 0.5),
    'std': (0.5, 0.5, 0.5),
    'embedding_dim': 512,
    'epochs': 5,
    'batch_size': 8,
    'lr': 2e-4,
    'weight_decay': 1e-4,
    'num_workers': 0,
    'val_ratio': 0.2,
    'min_images_per_identity': 3,
    'thresholds': [0.2, 0.25, 0.3, 0.35, 0.4],
    'seed': 42,
    'device': 'cuda',
    'train_subset': None,
    'val_subset': None,
    'lora_rank': 8,
    'lora_alpha': 16.0,
    'lora_dropout': 0.05,
    'lora_targets': ['qkv', 'proj'],
    'train_smoke_test': True,
}
CONFIG

{'zip_path': 'animal-clef-2026.zip',
 'output_dir': 'outputs/animal_clef_megadescriptor_lora',
 'backbone': 'hf-hub:BVRA/MegaDescriptor-L-384',
 'image_size': 384,
 'mean': (0.5, 0.5, 0.5),
 'std': (0.5, 0.5, 0.5),
 'embedding_dim': 512,
 'epochs': 5,
 'batch_size': 8,
 'lr': 0.0002,
 'weight_decay': 0.0001,
 'num_workers': 0,
 'val_ratio': 0.2,
 'min_images_per_identity': 3,
 'thresholds': [0.2, 0.25, 0.3, 0.35, 0.4],
 'seed': 42,
 'device': 'cuda',
 'train_subset': None,
 'val_subset': None,
 'lora_rank': 8,
 'lora_alpha': 16.0,
 'lora_dropout': 0.05,
 'lora_targets': ['qkv', 'proj'],
 'train_smoke_test': True}

In [ ]:
import io
import json
import math
import random
import zipfile
from dataclasses import dataclass

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from tqdm.auto import tqdm


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['seed'])
device = torch.device(CONFIG['device'] if torch.cuda.is_available() else 'cpu')
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)
print('device =', device)
print('output_dir =', output_dir)

In [ ]:
def load_metadata(zip_path: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as zf:
        with zf.open('metadata.csv') as f:
            df = pd.read_csv(f)
    for col in ['identity', 'path', 'species', 'split', 'dataset', 'orientation', 'date']:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str)
    df['is_labeled'] = df['identity'].ne('')
    return df


def make_train_val_split(metadata: pd.DataFrame, val_ratio: float, seed: int, min_images_per_identity: int):
    labeled = metadata[metadata['split'] == 'train'].copy()
    grouped = labeled.groupby('identity')
    rng = np.random.default_rng(seed)
    train_parts = []
    val_parts = []
    for _, group in grouped:
        group = group.sample(frac=1.0, random_state=int(rng.integers(0, 1_000_000))).reset_index(drop=True)
        if len(group) < min_images_per_identity:
            train_parts.append(group)
            continue
        val_count = max(1, int(round(len(group) * val_ratio)))
        val_count = min(val_count, len(group) - 1)
        val_parts.append(group.iloc[:val_count].copy())
        train_parts.append(group.iloc[val_count:].copy())
    train_df = pd.concat(train_parts, ignore_index=True).reset_index(drop=True)
    val_df = pd.concat(val_parts, ignore_index=True).reset_index(drop=True)
    return train_df, val_df


def maybe_subset(df: pd.DataFrame, limit: int | None, seed: int) -> pd.DataFrame:
    if limit is None or len(df) <= limit:
        return df.reset_index(drop=True)
    return df.sample(n=limit, random_state=seed).reset_index(drop=True)


metadata = load_metadata(CONFIG['zip_path'])
train_df, val_df = make_train_val_split(
    metadata,
    val_ratio=CONFIG['val_ratio'],
    seed=CONFIG['seed'],
    min_images_per_identity=CONFIG['min_images_per_identity'],
)
train_df = maybe_subset(train_df, CONFIG['train_subset'], CONFIG['seed'])
val_df = maybe_subset(val_df, CONFIG['val_subset'], CONFIG['seed'])
print('train images =', len(train_df))
print('val images =', len(val_df))
print('train identities =', train_df['identity'].nunique())
print(train_df.groupby('dataset')['identity'].nunique())

In [ ]:
class AnimalCLEFZipDataset(Dataset):
    def __init__(self, dataframe, zip_path, transform, label_to_idx=None, return_label=True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.zip_path = zip_path
        self.transform = transform
        self.label_to_idx = label_to_idx or {}
        self.return_label = return_label
        self._zip_file = None

    def __len__(self):
        return len(self.df)

    def _zf(self):
        if self._zip_file is None:
            self._zip_file = zipfile.ZipFile(self.zip_path)
        return self._zip_file

    def _load_image(self, relative_path: str):
        with self._zf().open(relative_path) as f:
            raw = f.read()
        return Image.open(io.BytesIO(raw)).convert('RGB')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = self._load_image(row['path'])
        image = self.transform(image)
        item = {
            'image': image,
            'image_id': int(row['image_id']),
            'identity': row['identity'],
            'dataset': row['dataset'],
            'species': row['species'],
            'path': row['path'],
        }
        if self.return_label:
            item['label'] = self.label_to_idx[row['identity']]
        return item


train_tf = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(CONFIG['mean'], CONFIG['std']),
])

eval_tf = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(CONFIG['mean'], CONFIG['std']),
])

labels = sorted(train_df['identity'].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
val_df = val_df[val_df['identity'].isin(label_to_idx)].reset_index(drop=True)

train_ds = AnimalCLEFZipDataset(train_df, CONFIG['zip_path'], train_tf, label_to_idx=label_to_idx, return_label=True)
val_ds = AnimalCLEFZipDataset(val_df, CONFIG['zip_path'], eval_tf, label_to_idx=label_to_idx, return_label=True)
val_embed_ds = AnimalCLEFZipDataset(val_df, CONFIG['zip_path'], eval_tf, label_to_idx=label_to_idx, return_label=False)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())
val_embed_loader = DataLoader(val_embed_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())

print('num_classes =', len(label_to_idx))

## LoRA Modules

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank if rank > 0 else 1.0
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.in_features = base_layer.in_features
        self.out_features = base_layer.out_features

        for p in self.base.parameters():
            p.requires_grad = False

        self.lora_A = nn.Parameter(torch.zeros(rank, self.in_features))
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.t()
        lora_out = lora_out @ self.lora_B.t()
        return base_out + self.scaling * lora_out


def replace_module_by_name(root: nn.Module, dotted_name: str, new_module: nn.Module):
    parts = dotted_name.split('.')
    parent = root
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], new_module)


def apply_lora_to_backbone(backbone: nn.Module, target_keywords, rank, alpha, dropout):
    replaced = []
    for name, module in list(backbone.named_modules()):
        if not isinstance(module, nn.Linear):
            continue
        if any(keyword in name for keyword in target_keywords):
            replace_module_by_name(backbone, name, LoRALinear(module, rank=rank, alpha=alpha, dropout=dropout))
            replaced.append(name)
    return replaced


def count_parameters(module: nn.Module):
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return {'total': total, 'trainable': trainable, 'frozen': total - trainable}


In [ ]:
class MegaDescriptorLoRAModel(nn.Module):
    def __init__(self, backbone_name, num_classes, embedding_dim, lora_rank, lora_alpha, lora_dropout, lora_targets):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool='avg')

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.lora_modules = apply_lora_to_backbone(
            self.backbone,
            target_keywords=lora_targets,
            rank=lora_rank,
            alpha=lora_alpha,
            dropout=lora_dropout,
        )

        feat_dim = getattr(self.backbone, 'num_features')
        self.projection = nn.Identity() if embedding_dim == feat_dim else nn.Linear(feat_dim, embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, images):
        features = self.backbone(images)
        projected = self.projection(features)
        logits = self.classifier(projected)
        embedding = F.normalize(projected, dim=1)
        return {'logits': logits, 'embedding': embedding, 'features': projected}


model = MegaDescriptorLoRAModel(
    backbone_name=CONFIG['backbone'],
    num_classes=len(label_to_idx),
    embedding_dim=CONFIG['embedding_dim'],
    lora_rank=CONFIG['lora_rank'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    lora_targets=CONFIG['lora_targets'],
).to(device)

param_info = count_parameters(model)
print('replaced LoRA modules =', len(model.lora_modules))
print(json.dumps(param_info, ensure_ascii=False, indent=2))
print(model.lora_modules[:20])

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
)
criterion = nn.CrossEntropyLoss()
optimizer

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader, device):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc='embed', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        embeddings = model(images)['embedding'].cpu().numpy()
        for i in range(len(batch['image_id'])):
            rows.append({
                'image_id': int(batch['image_id'][i]),
                'identity': batch['identity'][i],
                'dataset': batch['dataset'][i],
                'species': batch['species'][i],
                'path': batch['path'][i],
                'embedding': embeddings[i],
            })
    return pd.DataFrame(rows)


def cluster_embeddings(embeddings: np.ndarray, threshold: float):
    if len(embeddings) == 1:
        return np.array([0], dtype=int)
    model = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=threshold,
    )
    return model.fit_predict(embeddings)


def evaluate_clustering(embedding_df: pd.DataFrame, thresholds):
    thresholds = [float(t) for t in thresholds]
    threshold_scores = {}
    best_thresholds = {}
    best_dataset_scores = {}
    for dataset_name, group in embedding_df.groupby('dataset'):
        score_map = {}
        best_score = -1.0
        best_t = thresholds[0]
        if group['identity'].nunique() < 2:
            for t in thresholds:
                score_map[f'{t:.4f}'] = float('nan')
            best_thresholds[dataset_name] = best_t
            best_dataset_scores[dataset_name] = 0.0
            threshold_scores[dataset_name] = score_map
            continue

        embeddings = np.stack(group['embedding'].to_list())
        for t in thresholds:
            preds = cluster_embeddings(embeddings, t)
            score = float(adjusted_rand_score(group['identity'], preds))
            score_map[f'{t:.4f}'] = score
            if np.isfinite(score) and score > best_score:
                best_score = score
                best_t = float(t)
        threshold_scores[dataset_name] = score_map
        best_thresholds[dataset_name] = best_t
        best_dataset_scores[dataset_name] = best_score if best_score > -1.0 else 0.0
    mean_ari = float(np.mean(list(best_dataset_scores.values()))) if best_dataset_scores else 0.0
    return mean_ari, best_thresholds, threshold_scores, best_dataset_scores


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total = 0
    for batch in tqdm(loader, desc='train', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs['logits'], labels)
        loss.backward()
        optimizer.step()
        bs = images.size(0)
        total_loss += loss.item() * bs
        total += bs
    return total_loss / max(total, 1)


@torch.no_grad()
def eval_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total = 0
    for batch in tqdm(loader, desc='val-loss', leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs['logits'], labels)
        bs = images.size(0)
        total_loss += loss.item() * bs
        total += bs
    return total_loss / max(total, 1)

## Train

如果你只是先測流程，把 `CONFIG['train_smoke_test'] = True`，並把 subset 設小。

正式跑時可把 `train_subset / val_subset = None`。

In [ ]:
history = []
best_ari = -1.0
best_ckpt = output_dir / 'best.pt'

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = eval_loss(model, val_loader, criterion, device)
    embedding_df = extract_embeddings(model, val_embed_loader, device)
    val_ari, best_thresholds, threshold_scores, dataset_scores = evaluate_clustering(embedding_df, CONFIG['thresholds'])
    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_ari_mean': val_ari,
        'best_thresholds': best_thresholds,
        'per_dataset_best_ari': dataset_scores,
        'trainable_params': count_parameters(model)['trainable'],
    }
    history.append(row)
    print(json.dumps(row, ensure_ascii=False))

    if val_ari > best_ari:
        best_ari = val_ari
        torch.save({
            'model': model.state_dict(),
            'config': CONFIG,
            'label_to_idx': label_to_idx,
            'best_thresholds': best_thresholds,
            'best_ari': best_ari,
            'lora_modules': model.lora_modules,
        }, best_ckpt)

Path(output_dir / 'history.json').write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
print('best_ckpt =', best_ckpt)
print('best_ari =', best_ari)

In [ ]:
history

## Submission

In [ ]:
test_df = metadata[metadata['split'] == 'test'].copy().reset_index(drop=True)
test_ds = AnimalCLEFZipDataset(test_df, CONFIG['zip_path'], eval_tf, return_label=False)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available())

checkpoint = torch.load(best_ckpt, map_location='cpu', weights_only=False)
submission_model = MegaDescriptorLoRAModel(
    backbone_name=checkpoint['config']['backbone'],
    num_classes=len(checkpoint['label_to_idx']),
    embedding_dim=checkpoint['config']['embedding_dim'],
    lora_rank=checkpoint['config']['lora_rank'],
    lora_alpha=checkpoint['config']['lora_alpha'],
    lora_dropout=checkpoint['config']['lora_dropout'],
    lora_targets=checkpoint['config']['lora_targets'],
).to(device)
submission_model.load_state_dict(checkpoint['model'])
submission_model.eval()

test_embeddings = extract_embeddings(submission_model, test_loader, device)
threshold_map = checkpoint['best_thresholds']
default_threshold = float(CONFIG['thresholds'][0])
parts = []
for dataset_name, group in test_embeddings.groupby('dataset'):
    threshold = float(threshold_map.get(dataset_name, default_threshold))
    preds = cluster_embeddings(np.stack(group['embedding'].to_list()), threshold)
    out = group[['image_id']].copy()
    out['cluster'] = [f'cluster_{dataset_name}_{int(cid)}' for cid in preds]
    parts.append(out)
submission = pd.concat(parts, ignore_index=True).sort_values('image_id')
submission_path = output_dir / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('saved:', submission_path)
submission.head()

In [ ]:
meta = {
    'output_dir': str(output_dir),
    'best_ari': best_ari,
    'thresholds': checkpoint['best_thresholds'],
    'lora_modules': checkpoint['lora_modules'][:50],
}
(output_dir / 'run_meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
meta